# Esercitazione 2: Modellazione del Comfort Termico Urbano

**Modulo 2 — Deep Learning: Best Practice e Applicazioni**
Corso di Dottorato, Fondazione CIMA, 27 maggio 2026

In questo notebook costruiamo una rete neurale per predire un indice di comfort termico (UTCI) a partire da variabili meteorologiche e urbane.
Stesso stack del Notebook 1: Zarr + xarray, PyTorch Lightning, OmegaConf, MLFlow.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/meteocima/Monaco_DLCourse/blob/main/notebooks/02_urban_thermal_comfort_it.ipynb)

## 0. Setup

Usiamo [**uv**](https://docs.astral.sh/uv/) come gestore di pacchetti Python — un sostituto rapido e moderno di pip + virtualenv.

| Contesto | Comando | Cosa fa |
|----------|---------|--------|
| **Nuovo progetto** | `uv init` | Crea lo scaffold di un progetto con `pyproject.toml` |
| **Aggiungere una dipendenza** | `uv add torch xarray` | Aggiunge pacchetti a `pyproject.toml` e li installa |
| **Installare tutto** | `uv sync` | Crea un `.venv` e installa dal lockfile |
| **Eseguire un comando** | `uv run jupyter lab` | Esegue all'interno del `.venv` gestito |
| **Colab / Python di sistema** | `uv pip install --system -r requirements.txt` | Installa senza `.venv` (per ambienti non gestiti da te) |

Su **Colab** non c'è un `.venv` — Google fornisce l'ambiente Python — quindi usiamo `uv pip install --system`.

In [ ]:
import os

# On Colab: clone the repo (or pull latest) and install dependencies
if "COLAB_RELEASE_TAG" in os.environ:
    if os.path.exists("/content/Monaco_DLCourse"):
        !git -C /content/Monaco_DLCourse pull -q
    else:
        !git clone https://github.com/meteocima/Monaco_DLCourse.git /content/Monaco_DLCourse
    %cd /content/Monaco_DLCourse/notebooks
    !pip install -q uv
    !uv pip install --system -q -r ../requirements.txt

In [ ]:
from __future__ import annotations

from typing import Any

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import pytorch_lightning as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
import xarray as xr
import zarr
from omegaconf import DictConfig, OmegaConf
from torch.utils.data import DataLoader, Dataset

print(f"PyTorch version: {torch.__version__}")
print(f"Lightning version: {pl.__version__}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print(
        "\n⚠️  No GPU detected!\n"
        "   Go to Runtime → Change runtime type → T4 GPU\n"
        "   Then re-run this cell.\n"
        "   Training will still work on CPU, but will be slower."
    )

## 1. Configurazione con Hydra / OmegaConf

In produzione, Hydra carica file di configurazione YAML e li compone automaticamente tramite il suo decoratore `@hydra.main`. In un notebook usiamo **OmegaConf** (il backend di configurazione di Hydra) per caricare direttamente gli stessi file YAML — ottenendo una configurazione strutturata e tipizzata senza bisogno della CLI.

La funzione `load_config()` qui sotto carica il file YAML e permette di **sovrascrivere qualsiasi valore** tramite argomenti keyword — lo stesso pattern che Hydra usa da riga di comando (`max_epochs=50`), ma adattato per i notebook.

In [ ]:
def load_config(
    config_path: str = "../configs/urban_thermal.yaml",
    **overrides: Any,
) -> DictConfig:
    """Load a YAML config and apply keyword overrides.

    Flat shorthand keys (e.g. ``max_epochs=50``, ``batch_size=64``)
    are mapped to their nested config paths automatically via the
    ``SHORTCUTS`` table.

    Args:
        config_path: Path to the YAML configuration file.
        **overrides: Keyword overrides forwarded to the config.

    Returns:
        Merged OmegaConf ``DictConfig``.
    """
    cfg = OmegaConf.load(config_path)

    SHORTCUTS: dict[str, str] = {
        "max_epochs": "training.max_epochs",
        "learning_rate": "training.learning_rate",
        "batch_size": "data.batch_size",
        "hidden_sizes": "model.hidden_sizes",
        "dropout": "model.dropout",
        "time_start": "data.time_start",
        "time_end": "data.time_end",
        "train_split": "data.train_split",
    }
    for key, value in overrides.items():
        path = SHORTCUTS.get(key, key)
        OmegaConf.update(cfg, path, value)

    return cfg


cfg = load_config()
print(OmegaConf.to_yaml(cfg))

## 2. Dati: Variabili Climatiche Urbane ERA5

Carichiamo **dati reali di rianalisi ERA5** dal cloud Zarr store [ARCO-ERA5](https://github.com/google-research/arco-era5) — nessun download necessario! Lo store contiene l'intero archivio ERA5 a risoluzione 0.25°, ospitato su Google Cloud Storage. **Zarr** è un formato array chunked e compresso, ottimizzato per il cloud: `xr.open_zarr()` legge solo i *metadati* — i dati effettivi vengono recuperati in modo lazy quando si chiama `.compute()`.

### 📥 Variabili ERA5 grezze

| Variabile | Nome ERA5 | Unità |
|-----------|-----------|-------|
| Temperatura dell'aria a 2 m | `2m_temperature` | K |
| Temperatura di rugiada a 2 m | `2m_dewpoint_temperature` | K |
| Vento a 10 m (componente u) | `10m_u_component_of_wind` | m s⁻¹ |
| Vento a 10 m (componente v) | `10m_v_component_of_wind` | m s⁻¹ |
| Radiazione solare discendente al suolo | `surface_solar_radiation_downwards` | J m⁻² |
| Pressione media al livello del mare | `mean_sea_level_pressure` | Pa |

### 🔧 Feature derivate

Da queste deriviamo cinque predittori fisici:

| Feature | Formula | Unità |
|---------|---------|-------|
| $T_{2m}$ | `2m_temperature` − 273.15 | °C |
| $T_d$ | `2m_dewpoint_temperature` − 273.15 | °C |
| $v_{10}$ | $\sqrt{u_{10}^2 + v_{10}^2}$ | m s⁻¹ |
| $R_s$ | `surface_solar_radiation_downwards` (clipped ≥ 0) | J m⁻² |
| $P_{\text{msl}}$ | `mean_sea_level_pressure` / 100 | hPa |

### 🎯 Target: UTCI semplificato

L'[Universal Thermal Climate Index](https://utci.org/) integra temperatura, radiazione, umidità e vento in un'unica temperatura "percepita". Il calcolo completo dell'UTCI richiede un modello multi-nodo del corpo umano; qui usiamo un'**approssimazione semplificata** che cattura le principali dipendenze fisiche:

$$\text{UTCI} \approx 0.7\,T_{2m} + 0.2\,\frac{R_s}{100} - 0.5\,v_{10} - 0.3\,\Delta_h + 0.01\,T_{2m}\,\frac{R_s}{100}$$

dove $\Delta_h = 0.5\,(T_{2m} - T_d)$ è un termine di depressione dell'umidità.

**Interpretazione fisica di ciascun termine:**

- $0.7\,T_{2m}$ — la **temperatura dell'aria domina** il calore percepito
- $0.2\,R_s / 100$ — la **radiazione solare** aumenta lo stress termico (il sole caldo fa percepire più caldo)
- $-0.5\,v_{10}$ — effetto **wind chill**: il vento aumenta la dispersione convettiva del calore
- $-0.3\,\Delta_h$ — l'**aria secca** (grande $T_{2m} - T_d$) migliora il raffreddamento evaporativo
- $0.01\,T_{2m}\,R_s / 100$ — **interazione**: la radiazione conta *di più* quando fa già caldo

> **Nota:** L'apertura dei metadati ARCO-ERA5 può richiedere ~30–60 secondi al primo accesso.

In [ ]:
def load_data(cfg: DictConfig) -> dict[str, Any]:
    """Load ERA5 data for configured cities and derive features + UTCI.

    1. Opens the ARCO-ERA5 Zarr store lazily (metadata only).
    2. For each city in ``cfg.data.cities``, selects the nearest grid
       point and downloads the time range ``[time_start, time_end]``.
    3. Derives five physical features and the simplified UTCI target.

    Args:
        cfg: Configuration with ``data.zarr_url``, ``data.cities``,
            ``data.time_start``, and ``data.time_end``.

    Returns:
        Dict with keys ``features`` (N×5 float32 array),
        ``utci`` (N float32 array), ``feature_names`` (list[str]),
        and ``df`` (full pandas DataFrame).
    """
    surface_vars = [
        "2m_temperature",
        "2m_dewpoint_temperature",
        "10m_u_component_of_wind",
        "10m_v_component_of_wind",
        "surface_solar_radiation_downwards",
        "mean_sea_level_pressure",
    ]

    # Open the ARCO-ERA5 Zarr store (lazy — only metadata is read)
    print("Opening ARCO-ERA5 store (this may take ~30-60s on first access)...")
    ds_era5 = xr.open_zarr(
        cfg.data.zarr_url,
        chunks=None,
        storage_options=dict(token="anon"),
    )
    print(f"Store opened. Available variables: {len(ds_era5.data_vars)}")

    # Extract surface variables for each city
    records: list[pd.DataFrame] = []
    for city, coords in cfg.data.cities.items():
        print(f"Loading {city} ({coords.lat}°N, {coords.lon}°E)...")
        ds_city = (
            ds_era5[surface_vars]
            .sel(
                latitude=coords.lat,
                longitude=coords.lon,
                method="nearest",
            )
            .sel(time=slice(cfg.data.time_start, cfg.data.time_end))
            .compute()
        )
        df = ds_city.to_dataframe().reset_index()
        df["city"] = city
        records.append(df)

    df_all = pd.concat(records, ignore_index=True).dropna()
    print(f"\nTotal records: {len(df_all)}")

    # Derive features
    df_all["t2m_C"] = df_all["2m_temperature"] - 273.15
    df_all["d2m_C"] = df_all["2m_dewpoint_temperature"] - 273.15
    df_all["wind_speed"] = np.sqrt(
        df_all["10m_u_component_of_wind"] ** 2
        + df_all["10m_v_component_of_wind"] ** 2
    )
    df_all["solar_rad"] = df_all[
        "surface_solar_radiation_downwards"
    ].clip(lower=0)
    df_all["mslp_hPa"] = df_all["mean_sea_level_pressure"] / 100

    # Simplified UTCI approximation
    humidity_effect = 0.5 * (df_all["t2m_C"] - df_all["d2m_C"])
    df_all["utci"] = (
        0.7 * df_all["t2m_C"]
        + 0.2 * df_all["solar_rad"] / 100
        - 0.5 * df_all["wind_speed"]
        - 0.3 * humidity_effect
        + 0.01 * df_all["t2m_C"] * df_all["solar_rad"] / 100
    ).astype(np.float32)

    feature_names = [
        "t2m_C", "d2m_C", "wind_speed", "solar_rad", "mslp_hPa",
    ]
    features = df_all[feature_names].values.astype(np.float32)
    utci = df_all["utci"].values.astype(np.float32)

    print(f"Features shape: {features.shape}")
    print(f"Target (UTCI) shape: {utci.shape}")
    print(f"UTCI range: [{utci.min():.1f}, {utci.max():.1f}] °C")

    return {
        "features": features,
        "utci": utci,
        "feature_names": feature_names,
        "df": df_all,
    }


data = load_data(cfg)

### 📊 Visualizzazione dei dati grezzi

Prima dell'addestramento è importante ispezionare i dati. I grafici a dispersione qui sotto mostrano ciascuna feature rispetto al target UTCI. Questo aiuta a:

- **Individuare relazioni non lineari** che giustificano l'uso di una rete neurale rispetto alla semplice regressione lineare
- **Identificare outlier** o problemi di qualità dei dati
- **Farsi un'idea della rilevanza delle feature** — feature con un trend chiaro rispetto all'UTCI sono probabilmente predittori importanti

In [ ]:
def plot_raw_data(data: dict[str, Any]) -> None:
    """Scatter-plot each feature against the UTCI target.

    Produces one subplot per feature, coloured by UTCI value.

    Args:
        data: Dict returned by :func:`load_data` with keys
            ``features``, ``utci``, and ``feature_names``.
    """
    features: np.ndarray = data["features"]
    utci: np.ndarray = data["utci"]
    feature_names: list[str] = data["feature_names"]

    n_feat = len(feature_names)
    ncols = 3
    nrows = (n_feat + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4 * nrows))
    for i, (name, ax) in enumerate(zip(feature_names, axes.flat)):
        ax.scatter(
            features[:, i], utci,
            alpha=0.1, s=5, c=utci, cmap="RdYlBu_r",
        )
        ax.set_xlabel(name)
        ax.set_ylabel("UTCI (°C)")
        ax.set_title(f"{name} vs UTCI")

    for j in range(n_feat, nrows * ncols):
        axes.flat[j].set_visible(False)

    plt.suptitle("Feature–Target Relationships", fontsize=14)
    plt.tight_layout()
    plt.show()


plot_raw_data(data)

## 3. PyTorch Dataset e DataModule

### Perché separare Dataset dal Modello?

PyTorch segue un pattern di **separazione delle responsabilità**: il `Dataset` gestisce l'accesso ai dati (*qual è il campione i?*), il `DataLoader` gestisce il batching e lo shuffling, e il `Model` gestisce il forward pass e la loss. Questo rende ogni componente riutilizzabile e testabile indipendentemente.

### Il pattern `__getitem__`

Ogni campione restituito da `__getitem__(i)` è una tupla `(input, target)`:

```
features  (5,) ──────────→ input  (5,)    ← 5 predittori meteorologici
utci      (1,) ──────────→ target (1,)    ← temperatura "percepita" UTCI
```

### Normalizzazione Z-Score

A differenza del Notebook 1 (che usa lo scaling min-max), qui applichiamo la normalizzazione **Z-score** (standard) — più comune per dati tabulari:

$$x_{\text{norm}} = \frac{x - \mu}{\sigma}$$

Questo centra ogni feature a zero con varianza unitaria, il che aiuta gli ottimizzatori basati sul gradiente a convergere più velocemente — tutte le feature contribuiscono su una scala comparabile indipendentemente dalle loro unità originali (°C vs hPa vs J m⁻²).

> ⚠️ **Nota sul data leakage:** in questo notebook le statistiche di normalizzazione ($\mu$, $\sigma$) sono calcolate sull'intero dataset. Questo è accettabile perché il target UTCI è una formula *sintetica* (non una misura reale), quindi non c'è fuga di informazioni. In produzione, calcola sempre le statistiche solo sullo split di training.

### Split cronologico

I campioni sono suddivisi in training / validazione per **ordine cronologico**, non casualmente. I record meteorologici su ore consecutive sono altamente correlati (autocorrelazione temporale), quindi uno split casuale farebbe trapelare campioni quasi-duplicati tra i set, gonfiando i punteggi di validazione.

### 🔀 Shuffling e Batch Size

- **Training loader**: con shuffling — interrompe l'ordine temporale così i batch consecutivi sono decorrelati, riducendo il rumore del gradiente
- **Validation loader**: *senza* shuffling — garantisce un calcolo riproducibile delle metriche

L'addestramento usa il **mini-batch gradient descent**: invece di calcolare la loss sull'intero dataset (lento) o su un singolo campione (rumoroso), facciamo la media su un batch. Batch più grandi danno gradienti più lisci ma richiedono più memoria GPU.

In [ ]:
class UrbanClimateDataset(Dataset):
    """Wraps pre-normalised feature/target arrays as a PyTorch Dataset.

    Each sample is a ``(features, target)`` tuple of float32 tensors.
    """

    def __init__(
        self,
        features: np.ndarray,
        targets: np.ndarray,
    ) -> None:
        self.features = torch.from_numpy(features)
        self.targets = torch.from_numpy(targets).unsqueeze(1)

    def __len__(self) -> int:
        return len(self.features)

    def __getitem__(
        self, idx: int,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        return self.features[idx], self.targets[idx]


class UrbanClimateDataModule(pl.LightningDataModule):
    """Lightning DataModule for the urban thermal comfort task.

    Handles Z-score normalisation and chronological train/val splitting.
    Normalisation statistics (``feat_mean``, ``feat_std``,
    ``target_mean``, ``target_std``) are stored as attributes so that
    predictions can be denormalised later.
    """

    def __init__(
        self,
        features: np.ndarray,
        targets: np.ndarray,
        cfg: DictConfig,
    ) -> None:
        super().__init__()
        self.features = features
        self.targets = targets
        self.cfg = cfg
        # Compute normalisation stats on full data
        self.feat_mean: np.ndarray = features.mean(axis=0)
        self.feat_std: np.ndarray = features.std(axis=0)
        self.target_mean: float = float(targets.mean())
        self.target_std: float = float(targets.std())

    def setup(self, stage: str | None = None) -> None:
        """Normalise and split into train / val datasets."""
        feat_norm = (self.features - self.feat_mean) / self.feat_std
        tgt_norm = (
            (self.targets - self.target_mean) / self.target_std
        )

        n = len(feat_norm)
        n_train = int(n * self.cfg.data.train_split)
        self.train_ds = UrbanClimateDataset(
            feat_norm[:n_train], tgt_norm[:n_train],
        )
        self.val_ds = UrbanClimateDataset(
            feat_norm[n_train:], tgt_norm[n_train:],
        )

    def train_dataloader(self) -> DataLoader:
        return DataLoader(
            self.train_ds,
            batch_size=self.cfg.data.batch_size,
            shuffle=True,
        )

    def val_dataloader(self) -> DataLoader:
        return DataLoader(
            self.val_ds,
            batch_size=self.cfg.data.batch_size,
        )


def create_datamodule(
    data: dict[str, Any],
    cfg: DictConfig,
) -> UrbanClimateDataModule:
    """Create, set up, and return the DataModule from loaded data.

    Args:
        data: Dict returned by :func:`load_data`.
        cfg: Configuration with ``data.train_split`` and
            ``data.batch_size``.

    Returns:
        Ready-to-use :class:`UrbanClimateDataModule`.
    """
    dm = UrbanClimateDataModule(data["features"], data["utci"], cfg)
    dm.setup()
    print(f"Train: {len(dm.train_ds)}, Val: {len(dm.val_ds)}")
    return dm


dm = create_datamodule(data, cfg)

## 4. Modello: MLP per la Predizione del Comfort Termico

Un **Multi-Layer Perceptron (MLP)** — l'architettura deep learning più semplice. A differenza della CNN nel Notebook 1 (che sfrutta la struttura *spaziale*), l'MLP tratta ogni campione come un vettore piatto e apprende attraverso una pila di layer fully connected (densi).

### Architettura

```
Input (5,) → [Linear → ReLU → Dropout] × N → Linear → Output (1,)
```

### Layer Denso (Linear)

Ogni layer applica una matrice di pesi appresa e un bias:

$$\mathbf{h} = \sigma\!\left(\mathbf{W}\mathbf{x} + \mathbf{b}\right)$$

dove $\mathbf{x} \in \mathbb{R}^{d_{\text{in}}}$, $\mathbf{W} \in \mathbb{R}^{d_{\text{out}} \times d_{\text{in}}}$, $\mathbf{b} \in \mathbb{R}^{d_{\text{out}}}$, e $\sigma$ è un'attivazione non lineare.

### Funzione di attivazione — ReLU

$$\text{ReLU}(x) = \max(0, x)$$

Le attivazioni non lineari sono essenziali: senza di esse, impilare più layer lineari equivarrebbe a un'*unica* trasformazione lineare ($\mathbf{W}_2\mathbf{W}_1 = \mathbf{W}'$), e la rete non potrebbe apprendere relazioni non lineari complesse come la formula UTCI.

### Dropout — Regolarizzazione

$$\text{Dropout}(x_i) = \begin{cases} 0 & \text{con probabilità } p \\ x_i / (1-p) & \text{altrimenti}\end{cases}$$

Durante l'addestramento, il dropout **azzera casualmente** una frazione $p$ dei neuroni ad ogni forward pass. Questo previene la *co-adattazione*: i neuroni non possono fare affidamento su partner specifici, forzando la rete ad apprendere feature più robuste. In fase di inferenza il dropout è disabilitato (tutti i neuroni attivi).

### 🔌 Componenti iniettabili

Il modello accetta override opzionali per **loss function**, **optimizer** e **attivazione** — scambiali nel Playground per sperimentare:

| Componente | Default | Alternative da provare |
|------------|---------|------------------------|
| Loss | `nn.MSELoss()` | `nn.L1Loss()`, `nn.HuberLoss()` |
| Optimizer | `torch.optim.Adam` | `AdamW`, `SGD` |
| Attivazione | `nn.ReLU` | `nn.GELU`, `nn.SiLU`, `nn.LeakyReLU` |

> **Nota:** A differenza della CNN nel Notebook 1, questo MLP **non ha connessioni residue** — apprende la mappatura completa $\text{features} \to \text{UTCI}$ direttamente, invece di predire una correzione a una previsione esistente.

In [ ]:
class ThermalComfortMLP(pl.LightningModule):
    """MLP for predicting UTCI from meteorological features.

    The network is a configurable stack of
    ``[Linear → activation → Dropout]`` blocks followed by a final
    ``Linear(→ 1)`` output layer.  Loss function, optimizer, and
    activation are injectable for easy experimentation.
    """

    def __init__(
        self,
        cfg: DictConfig,
        n_features: int,
        loss_fn: nn.Module | None = None,
        optimizer_cls: type[torch.optim.Optimizer] | None = None,
        activation_cls: type[nn.Module] | None = None,
    ) -> None:
        super().__init__()
        self.save_hyperparameters(
            ignore=["loss_fn", "optimizer_cls", "activation_cls"],
        )
        self.cfg = cfg
        self.loss_fn = loss_fn or nn.MSELoss()
        self.optimizer_cls = optimizer_cls or torch.optim.Adam
        activation_cls = activation_cls or nn.ReLU

        layers: list[nn.Module] = []
        in_size = n_features
        for hidden_size in cfg.model.hidden_sizes:
            layers.extend([
                nn.Linear(in_size, hidden_size),
                activation_cls(),
                nn.Dropout(cfg.model.dropout),
            ])
            in_size = hidden_size
        layers.append(nn.Linear(in_size, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass: features → predicted UTCI (normalised)."""
        return self.net(x)

    def training_step(
        self,
        batch: tuple[torch.Tensor, torch.Tensor],
        batch_idx: int,
    ) -> torch.Tensor:
        """Compute training loss and log it."""
        x, y = batch
        pred = self(x)
        loss = self.loss_fn(pred, y)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(
        self,
        batch: tuple[torch.Tensor, torch.Tensor],
        batch_idx: int,
    ) -> torch.Tensor:
        """Compute validation loss and MAE, log both."""
        x, y = batch
        pred = self(x)
        loss = self.loss_fn(pred, y)
        mae = F.l1_loss(pred, y)
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_mae", mae)
        return loss

    def configure_optimizers(self) -> torch.optim.Optimizer:
        """Return the configured optimizer."""
        return self.optimizer_cls(
            self.parameters(), lr=self.cfg.training.learning_rate,
        )


def create_model(
    cfg: DictConfig,
    n_features: int,
    model_cls: type[pl.LightningModule] = ThermalComfortMLP,
    **model_kwargs: Any,
) -> pl.LightningModule:
    """Instantiate the MLP model from config.

    Args:
        cfg: Configuration with ``model.*`` keys.
        n_features: Number of input features.
        model_cls: Lightning module class to instantiate.
        **model_kwargs: Extra kwargs forwarded to the constructor
            (e.g. ``loss_fn``, ``optimizer_cls``, ``activation_cls``).

    Returns:
        Initialised Lightning module.
    """
    model = model_cls(cfg, n_features=n_features, **model_kwargs)
    print(model)
    return model


model = create_model(cfg, n_features=len(data["feature_names"]))

## 5. Addestramento con Lightning + MLFlow Tracking

Il `Trainer` di Lightning gestisce il loop di addestramento, il posizionamento su GPU e il logging. Avvolgiamo l'esecuzione in un **contesto MLFlow** così che ogni iperparametro e metrica vengano tracciati automaticamente. La funzione `train()` restituisce sia il trainer che un dizionario di metriche finali convertite in **°C** usando il fattore di scala della normalizzazione.

### 📉 Loss Function — MSE

Minimizziamo l'**Errore Quadratico Medio** tra predizioni e verità:

$$\mathcal{L}_{\text{MSE}} = \frac{1}{N}\sum_{i=1}^{N}\left(y_i - \hat{y}_i\right)^2$$

L'elevamento al quadrato penalizza gli errori grandi in modo sproporzionato — un errore di 10 °C contribuisce 100× di più di un errore di 1 °C. Le alternative includono il **MAE** (loss L1, più robusto agli outlier) e la **Huber loss** (combina MSE per errori piccoli con MAE per errori grandi).

### ⚡ Optimizer — Adam

**Adam** (Adaptive Moment Estimation) mantiene learning rate per-parametro usando stime mobili della media e varianza del gradiente. Intuizione: adatta la dimensione del passo per ogni dimensione — parametri con gradienti consistentemente grandi fanno passi più piccoli, e viceversa. Questo lo rende molto meno sensibile al learning rate iniziale rispetto al semplice SGD.

### 🎛️ Learning Rate

Il learning rate (default `1e-3`) controlla la dimensione del passo nello spazio dei parametri:
- **Troppo alto** → l'ottimizzatore supera i minimi, la loss oscilla o diverge
- **Troppo basso** → la convergenza è dolorosamente lenta, l'addestramento può rimanere bloccato in minimi locali scadenti

### 🔁 Epoche

Un'**epoca** = un passaggio completo attraverso tutti i campioni di training. Con 30 epoche, il modello vede ogni campione 30 volte, raffinando i suoi parametri ad ogni passaggio.

### 📊 Monitoraggio della validazione

Dopo ogni epoca il trainer calcola la loss sul **set di validazione** tenuto da parte. Questo rivela l'overfitting: se la loss di training continua a diminuire ma la loss di validazione inizia ad aumentare, il modello sta memorizzando i dati di training anziché apprendere pattern generalizzabili.

### 🔬 Integrazione MLFlow

Ogni chiamata a `train()` crea un *run* MLFlow all'interno dell'*esperimento* configurato. Il run registra:
1. **Tutti i parametri di configurazione** (tramite `mlflow.log_params`)
2. **Le metriche finali di validazione** in unità fisiche (°C)
3. Un **`run_id`** restituito al chiamante così che le metriche di valutazione possano essere aggiunte allo stesso run in seguito

In [ ]:
def train(
    model: pl.LightningModule,
    datamodule: UrbanClimateDataModule,
    cfg: DictConfig,
    target_std: float,
    run_name: str = "mlp_thermal_comfort",
) -> tuple[pl.Trainer, dict[str, float], str]:
    """Train the model with Lightning and log to MLFlow.

    Args:
        model: The MLP to train.
        datamodule: Lightning DataModule (already set up).
        cfg: Configuration with ``training.*`` and ``mlflow.*`` keys.
        target_std: Standard deviation of the UTCI target, used to
            convert normalised metrics back to °C.
        run_name: Name for the MLflow run (shown in the runs table).

    Returns:
        Tuple of ``(trainer, metrics, run_id)`` where *metrics*
        contains ``val_rmse_utci`` and ``val_mae_utci`` in °C, and
        *run_id* is the MLflow run ID for logging additional metrics
        later.
    """
    mlflow.set_experiment(cfg.mlflow.experiment_name)

    with mlflow.start_run(run_name=run_name) as run:
        mlflow.log_params(
            OmegaConf.to_container(cfg, resolve=True),
        )

        trainer = pl.Trainer(
            max_epochs=cfg.training.max_epochs,
            accelerator=cfg.training.accelerator,
            enable_progress_bar=True,
            log_every_n_steps=5,
        )
        trainer.fit(model, datamodule)

        metrics: dict[str, float] = {}
        val_loss = trainer.callback_metrics.get("val_loss")
        val_mae = trainer.callback_metrics.get("val_mae")
        if val_loss is not None:
            metrics["val_rmse_utci"] = float(
                val_loss.item() ** 0.5 * target_std,
            )
            mlflow.log_metric(
                "final_val_rmse_utci", metrics["val_rmse_utci"],
            )
        if val_mae is not None:
            metrics["val_mae_utci"] = float(
                val_mae.item() * target_std,
            )
            mlflow.log_metric(
                "final_val_mae_utci", metrics["val_mae_utci"],
            )

    rmse = metrics.get("val_rmse_utci")
    mae = metrics.get("val_mae_utci")
    if rmse is not None:
        print(f"\nFinal val RMSE: {rmse:.4f} °C")
    if mae is not None:
        print(f"Final val MAE:  {mae:.4f} °C")

    return trainer, metrics, run.info.run_id


trainer, metrics, run_id = train(model, dm, cfg, dm.target_std)

In [ ]:
def show_mlflow_runs(cfg: DictConfig) -> None:
    """Display a summary table of MLflow runs for the experiment.

    Queries the MLflow backend for all runs in
    ``cfg.mlflow.experiment_name`` and renders a tidy pandas
    DataFrame inline (no need for ``mlflow ui``).

    Args:
        cfg: Configuration with ``mlflow.experiment_name``.
    """
    from IPython.display import display

    experiment = mlflow.get_experiment_by_name(
        cfg.mlflow.experiment_name,
    )
    if experiment is None:
        print("No MLflow experiment found.")
        return

    df = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id],
        order_by=["start_time DESC"],
    )
    if df.empty:
        print("No runs logged yet.")
        return

    # Build a tidy summary
    cols: dict[str, Any] = {}
    cols["Run Name"] = df["tags.mlflow.runName"]
    cols["Status"] = df["status"]
    cols["Start"] = pd.to_datetime(
        df["start_time"],
    ).dt.strftime("%H:%M:%S")

    # Include any metrics columns present
    for col in sorted(df.columns):
        if col.startswith("metrics."):
            short = col.replace("metrics.", "")
            cols[short] = df[col]

    # Key hyperparameters
    for key in [
        "params.training.max_epochs",
        "params.training.learning_rate",
        "params.model.hidden_sizes",
        "params.model.dropout",
    ]:
        if key in df.columns:
            short = key.replace("params.", "")
            cols[short] = df[key]

    summary = pd.DataFrame(cols)
    display(summary)

## 6. Valutazione e Visualizzazione

La funzione `predict()` esegue l'inferenza sul set di validazione e **denormalizza** tutti gli output riportandoli in **°C** così che le metriche siano fisicamente significative. `evaluate()` poi genera un grafico a dispersione delle predizioni e un istogramma degli errori.

### Denormalizzazione

Per convertire le predizioni in UTCI °C invertiamo lo scaling Z-score:

$$\hat{y}_{\text{UTCI}} = \hat{y}_{\text{norm}} \cdot \sigma_y + \mu_y$$

Riportare le metriche in unità fisiche (°C) le rende direttamente interpretabili.

### 📐 Metriche

**RMSE** — Root Mean Squared Error (stesse unità della variabile, penalizza gli errori grandi):

$$\text{RMSE} = \sqrt{\frac{1}{N}\sum_{i=1}^{N}\left(y_i - \hat{y}_i\right)^2}$$

**MAE** — Mean Absolute Error (penalità lineare, più robusto agli outlier):

$$\text{MAE} = \frac{1}{N}\sum_{i=1}^{N}\left|y_i - \hat{y}_i\right|$$

**R²** — Coefficiente di Determinazione (frazione di varianza spiegata dal modello):

$$R^2 = 1 - \frac{\sum_{i=1}^{N}(y_i - \hat{y}_i)^2}{\sum_{i=1}^{N}(y_i - \bar{y})^2}$$

$R^2 = 1$ significa predizioni perfette; $R^2 = 0$ significa che il modello non è migliore del predire sempre la media.

### 📊 Interpretazione dei grafici

- **Grafico a dispersione** (predetto vs vero): punti vicini alla diagonale rossa = buone predizioni; deviazioni sistematiche rivelano bias
- **Istogramma degli errori**: centrato sullo zero = non distorto; distribuzione ampia = alta varianza

### 🔀 Importanza delle feature per permutazione

Per capire *quali feature contano di più*, **mescoliamo** una colonna di feature alla volta e rieseguiamo l'inferenza. Se mescolare una feature aumenta molto l'errore, quella feature era importante per predizioni accurate. Il punteggio di importanza è il rapporto $\text{MSE}_{\text{mescolato}} / \text{MSE}_{\text{baseline}}$ — valori vicini a 1.0 significano che la feature non conta; valori alti significano che è critica.

In [ ]:
def predict(
    model: pl.LightningModule,
    datamodule: UrbanClimateDataModule,
) -> dict[str, np.ndarray]:
    """Run inference on the validation set and denormalise to °C.

    Args:
        model: Trained MLP.
        datamodule: DataModule with ``val_ds`` populated and
            normalisation stats (``target_mean``, ``target_std``).

    Returns:
        Dict with ``pred_utci`` and ``true_utci`` (both in °C),
        and ``val_features`` (normalised tensors, kept for feature
        importance analysis).
    """
    model.eval()
    val_features: torch.Tensor = datamodule.val_ds.features
    val_targets: torch.Tensor = datamodule.val_ds.targets

    with torch.no_grad():
        val_pred = model(val_features)

    pred_utci: np.ndarray = (
        val_pred.numpy().squeeze()
        * datamodule.target_std
        + datamodule.target_mean
    )
    true_utci: np.ndarray = (
        val_targets.numpy().squeeze()
        * datamodule.target_std
        + datamodule.target_mean
    )

    return {
        "pred_utci": pred_utci,
        "true_utci": true_utci,
        "val_features": val_features,
    }


predictions = predict(model, dm)

In [ ]:
def evaluate(
    predictions: dict[str, np.ndarray],
) -> dict[str, float]:
    """Plot predicted vs true UTCI and error distribution.

    Args:
        predictions: Dict returned by :func:`predict` with
            ``pred_utci`` and ``true_utci`` (both in °C).

    Returns:
        Dict of evaluation metrics: ``eval_rmse_utci``,
        ``eval_mae_utci``, ``eval_r2``.
    """
    pred: np.ndarray = predictions["pred_utci"]
    true: np.ndarray = predictions["true_utci"]
    errors: np.ndarray = pred - true

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # Prediction scatter
    axes[0].scatter(true, pred, alpha=0.2, s=5)
    lims = [
        min(true.min(), pred.min()),
        max(true.max(), pred.max()),
    ]
    axes[0].plot(lims, lims, "r--", lw=2, label="Perfect prediction")
    axes[0].set_xlabel("True UTCI (°C)")
    axes[0].set_ylabel("Predicted UTCI (°C)")
    axes[0].set_title("Predicted vs True UTCI")
    axes[0].legend()

    # Error distribution
    axes[1].hist(errors, bins=50, edgecolor="black", alpha=0.7)
    axes[1].axvline(0, color="r", linestyle="--")
    axes[1].set_xlabel("Prediction Error (°C)")
    axes[1].set_ylabel("Count")
    axes[1].set_title(
        f"Error Distribution (MAE={np.mean(np.abs(errors)):.2f}°C)",
    )

    plt.tight_layout()
    plt.show()

    # Compute metrics
    rmse = float(np.sqrt(np.mean(errors**2)))
    mae = float(np.mean(np.abs(errors)))
    r2 = float(
        1 - np.sum(errors**2) / np.sum((true - true.mean()) ** 2),
    )
    print(f"RMSE: {rmse:.2f} °C")
    print(f"MAE:  {mae:.2f} °C")
    print(f"R²:   {r2:.4f}")

    return {
        "eval_rmse_utci": rmse,
        "eval_mae_utci": mae,
        "eval_r2": r2,
    }


eval_metrics = evaluate(predictions)

# Log evaluation metrics into the same MLflow run
with mlflow.start_run(run_id=run_id):
    mlflow.log_metrics(eval_metrics)

show_mlflow_runs(cfg)

In [ ]:
def feature_importance(
    model: pl.LightningModule,
    predictions: dict[str, Any],
    feature_names: list[str],
    target_std: float,
    target_mean: float,
) -> dict[str, float]:
    """Compute and plot permutation feature importance.

    For each feature, the column is randomly shuffled and inference
    is re-run.  The importance score is
    ``MSE_shuffled / MSE_baseline`` — higher means more important.

    Args:
        model: Trained MLP.
        predictions: Dict from :func:`predict` with
            ``val_features``, ``pred_utci``, and ``true_utci``.
        feature_names: List of feature names.
        target_std: Target σ for denormalisation.
        target_mean: Target μ for denormalisation.

    Returns:
        Dict mapping feature name → MSE ratio.
    """
    val_features: torch.Tensor = predictions["val_features"]
    true_utci: np.ndarray = predictions["true_utci"]
    pred_utci: np.ndarray = predictions["pred_utci"]

    baseline_mse: float = float(np.mean((pred_utci - true_utci) ** 2))
    importance: dict[str, float] = {}

    for i, name in enumerate(feature_names):
        feat_permuted = val_features.clone()
        perm_idx = torch.randperm(len(feat_permuted))
        feat_permuted[:, i] = feat_permuted[perm_idx, i]
        with torch.no_grad():
            perm_pred = (
                model(feat_permuted).numpy().squeeze()
                * target_std
                + target_mean
            )
        mse_permuted = float(np.mean((perm_pred - true_utci) ** 2))
        importance[name] = mse_permuted / baseline_mse

    # Plot
    names = list(importance.keys())
    values = list(importance.values())
    sorted_idx = np.argsort(values)[::-1]

    plt.figure(figsize=(8, 4))
    plt.barh(
        [names[i] for i in sorted_idx],
        [values[i] for i in sorted_idx],
    )
    plt.axvline(1.0, color="r", linestyle="--", label="Baseline")
    plt.xlabel("MSE ratio (higher = more important)")
    plt.title("Permutation Feature Importance")
    plt.legend()
    plt.tight_layout()
    plt.show()

    return importance


fi = feature_importance(
    model, predictions, data["feature_names"],
    dm.target_std, dm.target_mean,
)

## 7. Playground

Esegui l'intera pipeline in una cella. **Modifica i parametri qui sotto** e riesegui per sperimentare!

Puoi personalizzare:
- **Valori di configurazione** — `max_epochs`, `hidden_sizes`, `dropout`, `learning_rate`, `batch_size`
- **Intervallo temporale** — `time_start` / `time_end` (dati ERA5 disponibili dal `1940-01-01`)
- **Loss function** — sostituisci `nn.MSELoss()` con `nn.L1Loss()`, `nn.HuberLoss()`, ecc.
- **Optimizer** — prova `torch.optim.AdamW`, `torch.optim.SGD`, ecc.
- **Attivazione** — prova `nn.GELU`, `nn.SiLU`, `nn.LeakyReLU`, ecc.
- **Classe del modello** — definisci il tuo `LightningModule` e passalo tramite `model_cls`

In [ ]:
# ── 1. Config (change these!) ──────────────────────────────────
run_name = "mlp_thermal_comfort"  # ← change before each run!

cfg = load_config(
    max_epochs=30,
    hidden_sizes=[64, 32, 16],
    dropout=0.1,
    learning_rate=1e-3,
    batch_size=64,
)

# ── 2. Loss function ──────────────────────────────────────────
# Try: nn.L1Loss(), nn.HuberLoss(), nn.SmoothL1Loss()
loss_fn = nn.MSELoss()

# ── 3. Optimizer ───────────────────────────────────────────────
# Try: torch.optim.SGD, torch.optim.AdamW
optimizer_cls = torch.optim.Adam

# ── 4. Activation ─────────────────────────────────────────────
# Try: nn.GELU, nn.SiLU, nn.LeakyReLU
activation_cls = nn.ReLU

# ── 5. Run pipeline ───────────────────────────────────────────
data = load_data(cfg)
plot_raw_data(data)
dm = create_datamodule(data, cfg)
model = create_model(
    cfg,
    n_features=len(data["feature_names"]),
    loss_fn=loss_fn,
    optimizer_cls=optimizer_cls,
    activation_cls=activation_cls,
)
trainer, metrics, run_id = train(
    model, dm, cfg, dm.target_std, run_name=run_name,
)
predictions = predict(model, dm)
eval_metrics = evaluate(predictions)

# ── 6. Log evaluation metrics into MLflow ──────────────────────
with mlflow.start_run(run_id=run_id):
    mlflow.log_metrics(eval_metrics)

fi = feature_importance(
    model, predictions, data["feature_names"],
    dm.target_std, dm.target_mean,
)

# ── 7. Compare experiments ────────────────────────────────────
show_mlflow_runs(cfg)

## 8. Esercizi

Usa la cella **Playground** qui sopra per sperimentare — cambia un parametro alla volta e riesegui.

1. **Classi di comfort termico**: Converti l'UTCI in categorie di stress (nessuno stress, stress da calore moderato, stress da calore forte) e addestra un classificatore
2. **Ricerca dell'architettura**: Prova diverse dimensioni e profondità dei layer nascosti — usa il multirun di Hydra o sweep manuali
3. **Aggiungi batch normalisation**: Inserisci layer `nn.BatchNorm1d` e confronta le prestazioni
4. **Più città**: Aggiungi altre città europee a `cfg.data.cities` e riaddestra — il modello generalizza?
5. **Più variabili**: Carica ulteriori variabili ERA5 dallo store ARCO-ERA5 (es. `boundary_layer_height`, `soil_temperature_level_1`)
6. **Feature temporali**: Aggiungi ora del giorno e giorno dell'anno come feature cicliche (codifica sin/cos)
7. **Confronta gli esperimenti**: Chiama `show_mlflow_runs(cfg)` per vedere un riepilogo di tutti i tuoi run. Per l'interfaccia completa, gli utenti locali possono eseguire `!mlflow ui` e aprire `http://localhost:5000`